In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from tqdm.notebook import tqdm, trange
from torch import optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split


## 1. Data Loading and Preprocessing

In [ ]:
torch.manual_seed(42)

# 1. Define transforms
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])

# 2. Download the full datasets
full_train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transform)
full_test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transform)

# 3. Helper function to find the indices of the desired classes
def get_indices(dataset, classes=[0, 1]):
    # Create a boolean mask and extract the corresponding indices
    mask = torch.isin(dataset.targets, torch.tensor(classes))
    return mask.nonzero().squeeze().tolist()

# 4. Create filtered Subsets
binary_train_dataset = Subset(full_train_dataset, get_indices(full_train_dataset))
binary_test_dataset = Subset(full_test_dataset, get_indices(full_test_dataset))

# 5. Split train into train and validation sets using random_split
val_frac = 0.3
val_size = int(val_frac * len(binary_train_dataset))
train_size = len(binary_train_dataset) - val_size

train_subset, valid_subset = random_split(
    binary_train_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# 6. Create the DataLoaders
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_subset, batch_size=64, shuffle=False)
test_loader = DataLoader(binary_test_dataset, batch_size=64, shuffle=False)

print(f"MNIST train samples: {len(train_loader.dataset)}")
print(f"MNIST valid samples: {len(valid_loader.dataset)}")
print(f"MNIST test samples:  {len(test_loader.dataset)}")

## 2. Tropical MLP Implementation

In [ ]:
class TropLayer(nn.Module):
    """
    Tropical max-plus affine layer.
    Computes y_i = max_j (W_ij + x_j) with tropical bias via max(z, b).
    Bias is trainable and softly bounded with tanh scaling to avoid extremes.
    """
    
    def __init__(self, in_features: int, out_features: int, b_scale: float = 10.0):
        super().__init__()

        # Initialize weights with small random values to prevent large initial outputs
        self.W = nn.Parameter(torch.randn(out_features, in_features) * 0.1)

        # Initialize raw bias parameters to small negative values to start with low bias
        self.raw_b = nn.Parameter(torch.full((out_features,), -1.0))

        # Register b_scale as a buffer since it's a fixed hyperparameter, not learnable
        self.register_buffer("b_scale", torch.tensor(b_scale))

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # Compute the max-plus affine transformation: z_ij = W_ij + x_j
        z_all = self.W.unsqueeze(0) + x.unsqueeze(1)

        # Take the max over the input dimension to get z_i = max_j (W_ij + x_j)
        z_trop = torch.max(z_all, dim=2).values

        # Compute the bias term with tanh scaling to keep it in a reasonable range
        b = torch.tanh(self.raw_b) * self.b_scale

        # Combine the tropical output with the bias using max(z, b)
        return torch.max(z_trop, b.unsqueeze(0))


class TropMLP(nn.Module):
    """
    Tropical MLP built from TropLayer blocks.
    """

    def __init__(self, input_size: int, output_size: int, hidden_size: int = 128, num_layers: int = 1, b_scale: float = 10.0):
        super().__init__()

        self.flatten = nn.Flatten()
        self.first_hidden = TropLayer(input_size, hidden_size, b_scale=b_scale)
        self.hidden_layers = nn.ModuleList([
            TropLayer(hidden_size, hidden_size, b_scale=b_scale) for _ in range(num_layers - 1)
        ])
        self.outputlay = TropLayer(hidden_size, output_size, b_scale=b_scale)

    def forward(self, x):
        x = self.flatten(x)
        x = self.first_hidden(x)
        for layer in self.hidden_layers:
            x = layer(x)
        x = self.outputlay(x)
        return x

## 3. Definition of Training, Validation, and Testing Modules

### 3.1 Training

In [ ]:
def calculate_accuracy(y_pred, y):
    probs = torch.sigmoid(y_pred)
    preds = (probs >= 0.5).long()
    correct = preds.eq(y.long()).sum()
    acc = correct.float() / y.shape[0]
    return acc

In [ ]:
def train(model, iterator, optimizer, criterion, log_first_batches: int = 0):
    """
    Trains the model for one epoch using the provided data iterator.

    Args:
        model: the neural network model (e.g., TropMLP)
        iterator: DataLoader that yields batches of (x, y) for training
        optimizer: optimization algorithm (e.g., Adam)
        criterion: loss function (e.g., nn.MSELoss or nn.BCEWithLogitsLoss)
        log_first_batches: if >0, log mean prob on first N batches (debug)

    Returns:
        train_loss: weighted average loss over all training examples
        train_acc: average accuracy over all batches
    """

    epoch_loss = 0
    epoch_acc = 0

    model.train()

    for batch_idx, (x, y) in enumerate(tqdm(iterator, desc="Training", leave=False)):
        if y.dim() == 1:
            y = y.unsqueeze(1)
        y = y.to(dtype=x.dtype)

        optimizer.zero_grad()

        y_pred = model(x)
        loss = criterion(y_pred, y)
        acc = calculate_accuracy(y_pred, y)

        if log_first_batches > 0 and batch_idx < log_first_batches:
            probs = torch.sigmoid(y_pred)
            print(f"[train dbg] batch {batch_idx} prob mean={probs.mean().item():.3f} std={probs.std().item():.3f}")

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)


### 3.2 Validation

In [ ]:
def validate(model, iterator, criterion):
    """
    Evaluates the model on a validation set and returns the average loss.

    Args:
        model: PyTorch model to evaluate
        iterator: DataLoader for the validation data
        mse_loss_fn: loss function (nn.MSELoss or nn.BCEWithLogitsLoss)

    Returns:
        avg_loss: loss averaged over all validation examples
    """
    epoch_loss = 0
    epoch_acc = 0

    model.eval()

    with torch.no_grad():

        for (x, y) in tqdm(iterator, desc="Evaluating", leave=False):
            
            if y.dim() == 1:
                y = y.unsqueeze(1)
            y = y.to(dtype=x.dtype)

            y_pred = model(x)

            loss = criterion(y_pred, y)

            acc = calculate_accuracy(y_pred, y)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)


## 4. Setup and Configuration

In [ ]:
def epoch_time(start_time, end_time):
    """
    Calculates elapsed time between start and end timestamps.

    Args:
        start_time: float, start time in seconds
        end_time: float, end time in seconds

    Returns:
        elapsed_mins: integer number of minutes
        elapsed_secs: remaining seconds (after minutes are accounted for)
    """
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time // 60)
    elapsed_secs = int(elapsed_time % 60)
    return elapsed_mins, elapsed_secs

In [ ]:
model = TropMLP(
    input_size=784,     
    output_size=1,     
    hidden_size=32,
    num_layers=2,       
    b_scale=3
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

## 5. Training and Validation Loop

In [ ]:
EPOCHS = 150

best_valid_loss = float('inf')

results = []

for epoch in trange(EPOCHS, desc="Epochs"):

    start_time = time.monotonic()

    train_loss, train_acc = train(model, train_loader, optimizer, criterion, log_first_batches=0)
    valid_loss, valid_acc = validate(model, valid_loader, criterion)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'tropical_model.pt')
    end_time = time.monotonic()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    results.append({
        "Train Loss": train_loss,
        "Train Acc": train_acc,
        "Valid Loss": valid_loss,
        "Valid Acc": valid_acc})

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')


## 6. Results

### 6.1 Training and Validation Loss Curves

In [ ]:
def plot_results(results):

    plt.figure(figsize=(12, 5))
    plt.plot(results.index, results['Train Loss'], label='Train Loss', marker='o')
    plt.plot(results.index, results['Valid Loss'], label='Valid Loss', marker='s')
    plt.title('Training and Validation Loss', fontsize=16)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Loss', fontsize=14)
    plt.legend(fontsize=12)
    plt.xticks(results.index)
    plt.xticks(rotation=90)
    plt.grid(True)


    plt.figure(figsize=(12, 5))
    plt.plot(results.index, results['Train Acc'], label='Train Acc', marker='o')
    plt.plot(results.index, results['Valid Acc'], label='Valid Acc', marker='s')
    plt.title('Training and Validation Accuracy', fontsize=16)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Accuracy', fontsize=14)
    plt.legend(fontsize=12)
    plt.xticks(results.index)
    plt.xticks(rotation=90)
    plt.grid(True)


    plt.tight_layout()
    plt.show()

In [ ]:
df_results = pd.DataFrame(results, index=range(1, EPOCHS+1))
plot_results(df_results)

### 6.2 Test Results

In [ ]:
model.load_state_dict(torch.load('tropical_model.pt'))

model.eval()
test_loss, test_acc = validate(model, test_loader, criterion)
print(f'Test Loss: {test_loss:.3f}')
print(f'Test Accuracy: {test_acc*100:.2f}%')

### 6.3 Misclassified images 

In [ ]:
# Find and display some misclassified images with probabilities and logits
misclassified = []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        logits = model(x)
        probs = torch.sigmoid(logits).squeeze(1)
        preds = (probs >= 0.5).long()
        mism = preds != y
        if mism.any():
            for img, true_lbl, pred_lbl, prob, logit in zip(
                x[mism], y[mism], preds[mism], probs[mism], logits.squeeze(1)[mism]
            ):
                misclassified.append(
                    {
                        "img": img.cpu(),
                        "true": int(true_lbl.item()),
                        "pred": int(pred_lbl.item()),
                        "prob": float(prob.item()),
                        "logit": float(logit.item()),
                    }
                )
                if len(misclassified) >= 20:
                    break
        if len(misclassified) >= 20:
            break

print(f"Collected {len(misclassified)} misclassified images")

# Visualize the first n misclassified images
n_show = min(20, len(misclassified))
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
if n_show == 1:
    axes = [axes]
for ax, sample in zip(axes, misclassified[:n_show]):
    ax.imshow(sample["img"].squeeze().numpy(), cmap="gray")
    ax.axis("off")
    ax.set_title(
        f"true={sample['true']}\npred={sample['pred']}\nprob={sample['prob']:.3f}\nlogit={sample['logit']:.3f}",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

## 7. Generate LP constraints

In [ ]:
# Generate LP constraints for a specific test image
dataiter = iter(test_loader) 
images, labels = next(dataiter)

idx = 34
img_tensor = images[idx] 
label = labels[idx].item()

print(f"Image selected: {label}")

plt.imshow(img_tensor.squeeze(), cmap='gray')
plt.title(f"Label: {label}")
plt.show()

pixel_values = img_tensor.flatten().numpy()

print(f"\nNumber of pixels: {len(pixel_values)}")
print("Pixel values (first 20):", pixel_values[:20])

epsilon = 0.013
print("\n--- Generating constraints for LP ---")
for i, p in enumerate(pixel_values):

    lb = max(0.0, p - epsilon)
    ub = min(1.0, p + epsilon)
    
    print(f"x_{i} >= bias0 + {lb:.6f};")
    print(f"x_{i} <= bias0 + {ub:.6f};")

In [ ]:
with torch.no_grad():
    out = model(img_tensor.unsqueeze(0))
    print(f"Neural Network Output: {torch.sigmoid(out).item()}")